In [ ]:
import math
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone


# ============================================================
# 1. CONSTANTS
# ============================================================

# Astronomical unit
AU_KM = 149_597_870.7

# Earth radius, expressed in AU
R_EARTH_AU = 6378.137 / AU_KM

# Earth's axial tilt
OBLIQUITY = math.radians(23.43928)

# Uraniborg coordinates:
# latitude  = 55° 54' 28" N
# longitude = 12° 41' 48" E
LAT = math.radians(55 + 54/60 + 28/3600)
LON = math.radians(12 + 41/60 + 48/3600)

# Solar gravitational parameter in AU^3 / day^2.
# Chosen so that a = 1 AU gives Earth's sidereal year.
MU_SUN = 4 * math.pi**2 / (365.256363004**2)


# ============================================================
# 2. HIDDEN ORBIT PARAMETERS
# ============================================================
#
# These are used ONLY to generate the synthetic observations.
# Later, when reconstructing Mars's orbit, we pretend we
# do not know the Mars parameters.

EARTH = {
    "a": 1.00000011,                 # semi-major axis, AU
    "e": 0.01671022,                 # eccentricity
    "M0": math.radians(357.51716),   # initial mean anomaly
    "varpi": math.radians(102.93735) # perihelion direction
}

MARS = {
    "a": 1.52366231,
    "e": 0.09341233,
    "M0": math.radians(19.41248),
    "varpi": math.radians(336.04084)
}


# ============================================================
# 3. KEPLER EQUATION
# ============================================================

def solve_kepler(M, e, tol=1e-13, max_iter=50):
    """
    Solve Kepler's equation

        E - e sin(E) = M

    for the eccentric anomaly E.
    """

    E = float(M)

    for _ in range(max_iter):
        correction = (E - e * math.sin(E) - M) / \
                     (1 - e * math.cos(E))

        E -= correction

        if abs(correction) < tol:
            break

    return E


# ============================================================
# 4. ROTATION IN THE ORBITAL PLANE
# ============================================================

def rotate_z(v, angle):
    """
    Rotate a 3D vector around the z-axis.
    """

    c = math.cos(angle)
    s = math.sin(angle)

    R = np.array([
        [c, -s, 0.0],
        [s,  c, 0.0],
        [0.0, 0.0, 1.0]
    ])

    return R @ v


# ============================================================
# 5. PLANET POSITION AND VELOCITY
# ============================================================

def kepler_state(t_days, params):
    """
    Return heliocentric position and velocity for a planet.

    Position: AU
    Velocity: AU/day

    The orbit is a fixed Kepler ellipse.
    """

    a = params["a"]
    e = params["e"]

    # Mean angular speed
    n = math.sqrt(MU_SUN / a**3)

    # Mean anomaly
    M = params["M0"] + n * t_days

    # Solve Kepler's equation
    E = solve_kepler(M, e)

    # Position in orbital coordinates
    x = a * (math.cos(E) - e)

    y = (
        a
        * math.sqrt(1 - e**2)
        * math.sin(E)
    )

    r = np.array([x, y, 0.0])

    # Time derivative of eccentric anomaly
    Edot = n / (1 - e * math.cos(E))

    # Velocity
    vx = -a * math.sin(E) * Edot

    vy = (
        a
        * math.sqrt(1 - e**2)
        * math.cos(E)
        * Edot
    )

    v = np.array([vx, vy, 0.0])

    # Rotate so perihelion is not always on the x-axis
    r = rotate_z(r, params["varpi"])
    v = rotate_z(v, params["varpi"])

    return r, v


# ============================================================
# 6. DATE / TIME
# ============================================================

def days_from_j2000(dt):
    """
    Number of days from the J2000 reference epoch.
    """

    j2000 = datetime(
        2000, 1, 1, 12,
        tzinfo=timezone.utc
    )

    return (dt - j2000).total_seconds() / 86400.0


def planet_state(dt, params):

    t = days_from_j2000(dt)

    return kepler_state(t, params)


# ============================================================
# 7. ECLIPTIC -> EQUATORIAL COORDINATES
# ============================================================

def ecliptic_to_equatorial(v):
    """
    Account for Earth's axial tilt.
    """

    x, y, z = v

    c = math.cos(OBLIQUITY)
    s = math.sin(OBLIQUITY)

    return np.array([
        x,
        c*y - s*z,
        s*y + c*z
    ])


# ============================================================
# 8. EARTH ROTATION
# ============================================================

def gmst(dt):
    """
    Approximate Greenwich Mean Sidereal Time.

    Output: angle in radians.
    """

    D = days_from_j2000(dt)

    theta_deg = (
        280.46061837
        + 360.98564736629 * D
    ) % 360.0

    return math.radians(theta_deg)


# ============================================================
# 9. URANIBORG LOCAL COORDINATE SYSTEM
# ============================================================

def local_basis(dt):
    """
    East, North, Up directions at Uraniborg.
    """

    # Longitude shifts Greenwich sidereal time
    # to Uraniborg's local sidereal angle.
    theta = gmst(dt) + LON

    up = np.array([
        math.cos(LAT) * math.cos(theta),
        math.cos(LAT) * math.sin(theta),
        math.sin(LAT)
    ])

    east = np.array([
        -math.sin(theta),
        math.cos(theta),
        0.0
    ])

    north = np.array([
        -math.sin(LAT) * math.cos(theta),
        -math.sin(LAT) * math.sin(theta),
        math.cos(LAT)
    ])

    return east, north, up


def observer_geocentric_position(dt):
    """
    Position of Uraniborg relative to Earth's center.
    """

    _, _, up = local_basis(dt)

    return R_EARTH_AU * up


# ============================================================
# 10. RIGHT ASCENSION / DECLINATION
# ============================================================

def radec_from_unit(u):

    x, y, z = u

    ra = math.degrees(
        math.atan2(y, x)
    ) % 360.0

    dec = math.degrees(
        math.asin(np.clip(z, -1.0, 1.0))
    )

    return ra, dec


# ============================================================
# 11. ALTITUDE / AZIMUTH
# ============================================================

def altaz_from_unit(u, dt):

    east, north, up = local_basis(dt)

    e = np.dot(u, east)
    n = np.dot(u, north)
    z = np.dot(u, up)

    altitude = math.degrees(
        math.asin(np.clip(z, -1.0, 1.0))
    )

    # Azimuth measured clockwise from north
    azimuth = math.degrees(
        math.atan2(e, n)
    ) % 360.0

    return altitude, azimuth


# ============================================================
# 12. OBSERVATIONAL NOISE
# ============================================================

def add_angular_noise(u, sigma_arcmin, rng):
    """
    Add approximately isotropic angular noise.

    sigma_arcmin = 1, 2, or 5 arcminutes.
    """

    if sigma_arcmin == 0:
        return u.copy()

    # Find a direction perpendicular to line of sight.
    reference = np.array([0.0, 0.0, 1.0])

    if abs(np.dot(u, reference)) > 0.9:
        reference = np.array([0.0, 1.0, 0.0])

    e1 = np.cross(u, reference)
    e1 /= np.linalg.norm(e1)

    e2 = np.cross(u, e1)

    # arcminutes -> radians
    sigma = math.radians(
        sigma_arcmin / 60.0
    )

    a, b = rng.normal(
        0.0,
        sigma,
        size=2
    )

    noisy = u + a*e1 + b*e2

    return noisy / np.linalg.norm(noisy)


# ============================================================
# 13. GENERATE TYCHO-LIKE OBSERVATIONS
# ============================================================

def make_observation_table(
    noise_arcmin,
    step_days=10,
    seed=638
):

    rng = np.random.default_rng(seed)

    observations = []

    # A time is necessary because Earth rotates.
    current = datetime(
        1582, 1, 1, 22,
        tzinfo=timezone.utc
    )

    end = datetime(
        1600, 12, 31, 22,
        tzinfo=timezone.utc
    )

    while current <= end:

        # -----------------------------------------
        # Earth and Mars heliocentric positions
        # -----------------------------------------

        r_earth_ecl, _ = planet_state(
            current, EARTH
        )

        r_mars_ecl, _ = planet_state(
            current, MARS
        )

        # Convert to equatorial coordinates
        r_earth = ecliptic_to_equatorial(
            r_earth_ecl
        )

        r_mars = ecliptic_to_equatorial(
            r_mars_ecl
        )

        # -----------------------------------------
        # Actual observer is on Earth's surface
        # -----------------------------------------

        r_site = observer_geocentric_position(
            current
        )

        r_observer = r_earth + r_site

        # -----------------------------------------
        # Observer -> Mars line of sight
        # -----------------------------------------

        line_of_sight = r_mars - r_observer

        u = (
            line_of_sight
            / np.linalg.norm(line_of_sight)
        )

        # -----------------------------------------
        # Check whether Mars is actually visible
        # -----------------------------------------

        mars_altitude, _ = altaz_from_unit(
            u, current
        )

        # Direction from observer toward Sun
        sun_direction = -r_observer
        sun_direction /= np.linalg.norm(
            sun_direction
        )

        sun_altitude, _ = altaz_from_unit(
            sun_direction,
            current
        )

        # Require Mars reasonably above horizon
        # and the Sun below the horizon.
        if (
            mars_altitude >= 10.0
            and sun_altitude <= -6.0
        ):

            # Add Tycho-like measurement error
            u_observed = add_angular_noise(
                u,
                noise_arcmin,
                rng
            )

            ra, dec = radec_from_unit(
                u_observed
            )

            alt, az = altaz_from_unit(
                u_observed,
                current
            )

            observations.append({
                "datetime_utc":
                    current.isoformat(),

                "ra_deg": ra,
                "dec_deg": dec,

                "alt_deg": alt,
                "az_deg": az,

                "noise_arcmin":
                    noise_arcmin
            })

        current += timedelta(days=step_days)

    return pd.DataFrame(observations)


# ============================================================
# 14. CONSERVATION-LAW VERIFICATION
# ============================================================

def verify_orbit(params):

    energies = []
    angular_momenta = []

    current = datetime(
        1582, 1, 1,
        tzinfo=timezone.utc
    )

    end = datetime(
        1600, 12, 31,
        tzinfo=timezone.utc
    )

    while current <= end:

        r, v = planet_state(
            current,
            params
        )

        radius = np.linalg.norm(r)

        # Specific orbital energy:
        #
        # E = 1/2 |v|^2 - mu/r
        #
        energy = (
            0.5 * np.dot(v, v)
            - MU_SUN / radius
        )

        # Specific angular momentum:
        #
        # h = r x v
        #
        h = np.linalg.norm(
            np.cross(r, v)
        )

        energies.append(energy)
        angular_momenta.append(h)

        current += timedelta(days=30)

    energies = np.array(energies)

    angular_momenta = np.array(
        angular_momenta
    )

    print(
        "Energy relative variation:",
        (
            energies.max()
            - energies.min()
        )
        / abs(energies.mean())
    )

    print(
        "Angular momentum relative variation:",
        (
            angular_momenta.max()
            - angular_momenta.min()
        )
        / abs(angular_momenta.mean())
    )


# ============================================================
# 15. CREATE THE DATA FILES
# ============================================================

for noise in [1, 2, 5]:

    table = make_observation_table(
        noise_arcmin=noise
    )

    filename = (
        f"synthetic_mars_{noise}arcmin.csv"
    )

    table.to_csv(
        filename,
        index=False
    )

    print(
        filename,
        len(table),
        "observations"
    )


print("\nEARTH CHECK")
verify_orbit(EARTH)

print("\nMARS CHECK")
verify_orbit(MARS)